# Phase 9 — RAGAS Evaluation of Sanctions RAG Output

This notebook evaluates the RAG triage output from **Notebook 08** using
[RAGAS](https://docs.ragas.io/) (Retrieval-Augmented Generation Assessment).

## What it does

1. **Securely collects** your Anthropic API key (censored on entry — never stored or printed)
2. **Loads** the enriched Excel file produced by Notebook 08
3. **Reconstructs** the RAGAS evaluation dataset (question, answer, contexts)
4. **Runs RAGAS metrics** using Claude as the evaluator LLM:
   - **Faithfulness** — Does the triage note only use facts from the retrieved records?
   - **Answer Relevancy** — Does the triage note address the sanctions query?
   - **Context Precision** — Are the most relevant records ranked highest?
   - **Context Recall** — Did the retrieval capture the records needed for the query?
5. **Adds sanctions-specific evaluation** on top of RAGAS:
   - Entity attribution accuracy
   - Sanctions programme grounding
   - Classification justification quality
6. **Exports** all scores back into the enriched Excel with a RAGAS summary sheet

## Requirements

```
pip install ragas datasets langchain-anthropic pandas openpyxl
```

- An **Anthropic API key** (entered at runtime, never stored)
- The **enriched Excel file** from Notebook 08

## Security

- The API key is collected via `getpass` (no echo)
- Only the first 4 and last 4 characters are ever displayed
- The key is held only in memory, never written to disk, logs, or cell output

---
## 0 · Configuration

In [3]:
# ======================================================================
# CONFIGURATION
# ======================================================================

# Auto-detects the most recent *_RAG_enriched.xlsx if set to None
RAG_OUTPUT_FILE = None

SHEET_NAME = 'rag_enriched'
EVAL_MODEL = 'claude-sonnet-4-20250514'
MAX_RECORDS_PER_QUERY = 10
RAGAS_SEED = 42

print('Configuration loaded.')
print(f'  Input file  : {RAG_OUTPUT_FILE or "(auto-detect)"}')
print(f'  Eval model  : {EVAL_MODEL}')


Configuration loaded.
  Input file  : rag_output/crazy_test_enriched_pooling_rag_FULL_20260407_RAG_enriched.xlsx
  Eval model  : claude-sonnet-4-20250514


---
## 1 · Secure API Key Entry

Your API key is:
- Collected via `getpass` (no on-screen echo while typing)
- Displayed only as `sk-a****...****wxyz` (first 4 + last 4 chars)
- **Never** written to disk, cell output, logs, or environment files
- Held only in memory for the duration of this session

In [5]:
# ======================================================================
# SECURE API KEY COLLECTION
# ======================================================================
import getpass
import os

def censor_api_key(key: str) -> str:
    if not key or len(key) < 12:
        return '****INVALID_KEY_FORMAT****'
    return key[:4] + "*" * 20 + "..." + "*" * 8 + key[-4:]

print("Enter your Anthropic API key below.")
print("(Input is hidden)")
print()
_api_key = getpass.getpass(prompt='Anthropic API Key: ')

if not _api_key or len(_api_key) < 20:
    raise ValueError("API key appears too short.")

print(f"
  Key received: {censor_api_key(_api_key)}")
os.environ['ANTHROPIC_API_KEY'] = _api_key
del _api_key
print("  API key is ready.")


Enter your Anthropic API key below.
(Input is hidden — nothing will appear as you type)



Anthropic API Key:  ········



  Key received: sk-a********************...********MwAA
  Key length  : 108 characters
  Status      : Stored in memory only (not written to disk)

  API key is ready for use via environment variable.


---
## 2 · Imports & Dependency Check

In [7]:
# ======================================================================
# IMPORTS
# ======================================================================
import sys
import re
import json
import time
import warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

# --- Dependency check ---
missing = []
for pkg in ['ragas', 'datasets', 'langchain_anthropic']:
    try:
        __import__(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print("Missing packages detected. Install with:")
    print(f"  pip install {' '.join(m.replace('_', '-') for m in missing)}")
    print("\nThen restart the kernel and re-run.")
    raise ImportError(f"Missing: {missing}")

# RAGAS imports
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic

print("All imports loaded successfully.")
print(f"  ragas version     : {__import__('ragas').__version__}")
print(f"  datasets version  : {__import__('datasets').__version__}")

All imports loaded successfully.
  ragas version     : 0.4.3
  datasets version  : 4.8.4


---
## 3 · Initialise Anthropic Evaluator LLM

RAGAS uses an LLM to judge the quality of RAG outputs. We configure Claude
as the evaluator via `langchain-anthropic`.

In [12]:
# ======================================================================
# EVALUATOR LLM — Claude via Anthropic
# ======================================================================

eval_llm = ChatAnthropic(
    model=EVAL_MODEL,
    temperature=0.0,
    max_tokens=4096,
    # API key is read automatically from ANTHROPIC_API_KEY env var
)

# Wrap for RAGAS compatibility
ragas_llm = LangchainLLMWrapper(eval_llm)

# Quick connectivity test (minimal token usage)
try:
    test_resp = eval_llm.invoke("Reply with only the word: connected")
    print(f"Evaluator LLM ready: {EVAL_MODEL}")
    print(f"  Test response: {test_resp.content[:50]}")
except Exception as e:
    print(f"ERROR connecting to Anthropic API: {e}")
    print("Check your API key and network connection.")
    raise

ERROR connecting to Anthropic API: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZrubPu9BmBrziKmW2FJn'}
Check your API key and network connection.


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011CZrubPu9BmBrziKmW2FJn'}

---
## 4 · Load RAG Output from Notebook 08

Reads the enriched Excel file. We need:
- `query_id`, `query_text` — the original sanctions query
- `rag_note` — the generated triage note (the "answer")
- `rag_classification` — the LLM's classification
- Text/context columns — the retrieved records ("contexts")

In [ ]:
# ======================================================================
# Load the RAG-enriched Excel file from 08_b script/notebook
# ======================================================================

rag_path = None

if RAG_OUTPUT_FILE is not None:
    rag_path = Path(RAG_OUTPUT_FILE)
else:
    # Auto-detect: find the most recent *_RAG_enriched.xlsx
    search_dirs = [
        Path('../data/rag_output'),
        Path('data/rag_output'),
        Path('../rag_output'),
        Path('rag_output'),
    ]
    for search_dir in search_dirs:
        if search_dir.exists():
            candidates = sorted(search_dir.glob('*_RAG_enriched.xlsx'))
            if candidates:
                rag_path = candidates[-1]  # most recent
                break

if rag_path is None or not rag_path.exists():
    raise FileNotFoundError(
        'Could not find *_RAG_enriched.xlsx.\n'
        '  Run: python scripts/08_rag_result_test_script.py\n'
        '  Or set RAG_OUTPUT_FILE in the Configuration cell.'
    )

df = pd.read_excel(rag_path, sheet_name=SHEET_NAME)

print(f'Loaded: {rag_path}')
print(f'  Rows    : {len(df):,}')
print(f'  Columns : {len(df.columns)}')
print(f'  Queries : {df["query_id"].nunique()}')
print()

# Derive query_type if not present
if 'query_type' not in df.columns:
    df['query_type'] = df['query_id'].str.extract(r'Q(\d+)_')[0].apply(lambda x: f'Type {x}')
if 'query_notes' not in df.columns:
    df['query_notes'] = ''

# Show RAG-specific columns
rag_cols = [c for c in df.columns if c.startswith('rag_')]
print(f'RAG columns found: {rag_cols}')

# Quick data quality check
has_notes = (df['rag_note'].astype(str) != '').sum()
has_class = (df['rag_classification'].astype(str) != '').sum()
print(f'\n  Rows with RAG notes         : {has_notes:,} / {len(df):,}')
print(f'  Rows with classifications   : {has_class:,} / {len(df):,}')


---
## 5 · Build RAGAS Evaluation Dataset

RAGAS expects a HuggingFace `Dataset` with these columns:
- `question` — the original query
- `answer` — the generated RAG output
- `contexts` — list of retrieved text chunks
- `ground_truth` (optional) — analyst-verified answer

We reconstruct these from the Notebook 08 output by grouping records per `query_id`.

In [ ]:
# ======================================================================
# Build the RAGAS dataset — one row per query_id
# ======================================================================

def build_context_from_row(row):
    """
    Reconstruct the context string for a single retrieved record,
    matching the column names from 08_b script output.
    """
    parts = []

    caption = str(row.get('caption', ''))
    if caption and caption != 'nan':
        parts.append(f'Entity: {caption}')

    country = str(row.get('meta_country', ''))
    if country and country != 'nan':
        parts.append(f'Country: {country}')

    program = str(row.get('meta_programId', ''))
    if program and program != 'nan':
        parts.append(f'Programme: {program}')

    schema = str(row.get('schema', ''))
    if schema and schema != 'nan':
        parts.append(f'Type: {schema}')

    text = str(row.get('embedding_text', row.get('text_blob', '')))
    if text and text != 'nan':
        parts.append(f'Details: {text[:500]}')

    score = row.get('rrf_score', None)
    if score and not pd.isna(score):
        parts.append(f'RRF score: {score:.6f}')

    datasets = str(row.get('meta_datasets', ''))
    if datasets and datasets != 'nan':
        parts.append(f'Listed in: {datasets[:200]}')

    return ' | '.join(parts) if parts else 'No context available'


# Group by query_id and build RAGAS rows
ragas_rows = []
query_ids = df['query_id'].unique()

for qid in query_ids:
    query_group = df[df['query_id'] == qid].head(MAX_RECORDS_PER_QUERY)
    first_row = query_group.iloc[0]

    # Question
    question = str(first_row.get('query_text', ''))
    query_type = str(first_row.get('query_type', ''))
    query_notes = str(first_row.get('query_notes', ''))
    if query_notes and query_notes not in ('nan', ''):
        question = f'{question} [Type: {query_type}] [Notes: {query_notes}]'
    elif query_type and query_type != 'nan':
        question = f'{question} [Type: {query_type}]'

    # Answer — the generated triage note
    notes = query_group['rag_note'].astype(str)
    non_empty = notes[notes != ''].tolist()
    answer = '\n\n'.join(non_empty) if non_empty else 'No triage note generated.'

    # Contexts
    contexts = [build_context_from_row(row) for _, row in query_group.iterrows()]

    # Ground truth
    gt_parts = []
    for _, row in query_group.iterrows():
        caption = str(row.get('caption', ''))
        program = str(row.get('meta_programId', ''))
        country = str(row.get('meta_country', ''))
        if caption and caption != 'nan':
            gt = caption
            if program and program != 'nan':
                gt += f' — {program}'
            if country and country != 'nan':
                gt += f' ({country})'
            gt_parts.append(gt)
    ground_truth = '; '.join(gt_parts) if gt_parts else question

    ragas_rows.append({
        'question': question,
        'answer': answer,
        'contexts': contexts,
        'ground_truth': ground_truth,
        'query_id': str(qid),
    })

print(f'RAGAS dataset built: {len(ragas_rows)} queries')
print(f'\nSample entry:')
sample = ragas_rows[0]
print(f'  Question : {sample["question"][:80]}...')
print(f'  Answer   : {sample["answer"][:80]}...')
print(f'  Contexts : {len(sample["contexts"])} retrieved records')
print(f'  Ground T : {sample["ground_truth"][:80]}...')


---
## 6 · Run RAGAS Evaluation

Evaluates the RAG output with four core RAGAS metrics:

| Metric | What it measures | Sanctions relevance |
|---|---|---|
| **Faithfulness** | Are all claims grounded in the contexts? | Critical — hallucinated sanctions data is dangerous |
| **Answer Relevancy** | Does the answer address the query? | Does the triage note actually screen the queried entity? |
| **Context Precision** | Are the best contexts ranked first? | Are the most relevant sanctions records surfaced on top? |
| **Context Recall** | Did retrieval capture needed info? | Were all relevant sanctions programmes/designations found? |

> **Cost estimate:** ~$0.01–0.05 per query depending on context length.

In [22]:
# ======================================================================
# RAGAS EVALUATION
# ======================================================================

# Build HuggingFace Dataset (RAGAS requirement)
hf_dataset = Dataset.from_dict({
    'question': [r['question'] for r in ragas_rows],
    'answer': [r['answer'] for r in ragas_rows],
    'contexts': [r['contexts'] for r in ragas_rows],
    'ground_truth': [r['ground_truth'] for r in ragas_rows],
})

print(f"HuggingFace Dataset ready: {len(hf_dataset)} rows")
print(f"Columns: {hf_dataset.column_names}")
print()

# Define metrics
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
]

print(f"Running RAGAS evaluation with {len(metrics)} metrics...")
print(f"  Evaluator: {EVAL_MODEL}")
print(f"  Queries  : {len(hf_dataset)}")
print(f"  This may take a few minutes depending on query count.")
print()

t0 = time.time()

try:
    ragas_result = evaluate(
        dataset=hf_dataset,
        metrics=metrics,
        llm=ragas_llm,
        raise_exceptions=False,
    )
    ragas_elapsed = time.time() - t0
    
    print(f"\nRAGAS evaluation complete in {ragas_elapsed:.1f}s")
    print(f"\n{'='*60}")
    print("  RAGAS AGGREGATE SCORES")
    print(f"{'='*60}")
    for metric_name, score in ragas_result.items():
        if isinstance(score, (int, float)):
            print(f"  {metric_name:25s}: {score:.4f}")
    print(f"{'='*60}")
    
except Exception as e:
    ragas_elapsed = time.time() - t0
    print(f"\nRAGAS evaluation encountered an error after {ragas_elapsed:.1f}s:")
    print(f"  {type(e).__name__}: {e}")
    print("\nThis may be due to:")
    print("  - API rate limiting (wait and retry)")
    print("  - RAGAS version incompatibility (check ragas docs)")
    print("  - Data format issues (check cell 5 output)")
    ragas_result = None

NameError: name 'Dataset' is not defined

---
## 7 · Per-Query RAGAS Scores

In [ ]:
# ======================================================================
# Extract per-query scores from RAGAS result
# ======================================================================

if ragas_result is not None:
    # RAGAS returns a Dataset with per-row scores
    df_ragas = ragas_result.to_pandas()
    
    # Add query_id back
    df_ragas['query_id'] = [r['query_id'] for r in ragas_rows]
    
    # Reorder columns
    score_cols = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
    available_scores = [c for c in score_cols if c in df_ragas.columns]
    
    display_cols = ['query_id'] + available_scores
    display_cols = [c for c in display_cols if c in df_ragas.columns]
    
    print("PER-QUERY RAGAS SCORES:")
    print("-" * 80)
    print(df_ragas[display_cols].to_string(index=False, float_format='{:.4f}'.format))
    print()
    
    # Sanctions-relevant flags
    print("\nSANCTIONS RISK FLAGS:")
    print("-" * 80)
    for _, row in df_ragas.iterrows():
        qid = row.get('query_id', '?')
        faith = row.get('faithfulness', None)
        if faith is not None and faith < 0.7:
            print(f"  ⚠ {qid}: Faithfulness={faith:.4f} — HIGH RISK of hallucinated sanctions data")
        relevancy = row.get('answer_relevancy', None)
        if relevancy is not None and relevancy < 0.5:
            print(f"  ⚠ {qid}: Answer Relevancy={relevancy:.4f} — triage note may not address query")
    
    print("\nDone.")
else:
    print("RAGAS results not available. See errors above.")
    df_ragas = pd.DataFrame()

---
## 8 · Sanctions-Specific Evaluation Layer

RAGAS gives us general RAG quality metrics. This section adds **sanctions-domain**
evaluation using Claude to check:

1. **Entity Attribution** — Does every entity claim cite a specific record?
2. **Programme Grounding** — Are sanctions programmes mentioned only if present in contexts?
3. **Classification Justification** — Is the TRUE POSITIVE / FALSE POSITIVE / AMBIGUOUS 
   classification backed by sufficient field-level evidence?
4. **Fabrication Detection** — Are there any facts not traceable to any retrieved record?

In [ ]:
# ======================================================================
# SANCTIONS-SPECIFIC EVALUATION PROMPTS
# ======================================================================

SANCTIONS_EVAL_SYSTEM = """You are a sanctions compliance auditor reviewing an automated 
screening triage note. Your job is to verify that the triage note is:
1. Faithful to the retrieved records (no fabricated data)
2. Properly attributing entities and sanctions programmes
3. Providing justified classifications

You must be extremely strict. In sanctions compliance, a hallucinated 
entity name, fabricated programme, or unjustified TRUE POSITIVE 
classification can have severe legal and regulatory consequences.

Respond ONLY in the JSON format specified."""

SANCTIONS_EVAL_TEMPLATE = """Audit the following triage note against the retrieved records.

QUERY: {question}

RETRIEVED RECORDS:
{contexts}

TRIAGE NOTE TO AUDIT:
{answer}

---

Produce a JSON object with exactly these fields:

{{
  "entity_attribution_score": <float 0-1>,
  "entity_attribution_issues": ["<issue1>", ...],
  "programme_grounding_score": <float 0-1>,
  "programme_grounding_issues": ["<issue1>", ...],
  "classification_justification_score": <float 0-1>,
  "classification_justification_issues": ["<issue1>", ...],
  "fabrication_detected": <true/false>,
  "fabricated_claims": ["<claim1>", ...],
  "overall_sanctions_quality": <float 0-1>,
  "risk_level": "<LOW/MEDIUM/HIGH/CRITICAL>",
  "summary": "<1-2 sentence summary of audit findings>"
}}

Scoring guidance:
- 1.0 = Perfect, every claim traceable to records
- 0.7-0.9 = Minor issues (e.g. one uncited claim that is inferable)
- 0.4-0.6 = Significant issues (e.g. uncited entities, weak justification)
- 0.0-0.3 = Critical issues (fabricated data, phantom records)

Risk levels:
- LOW: All claims grounded, classifications justified
- MEDIUM: Minor attribution gaps but no fabrication
- HIGH: Ungrounded claims or weak classification justification
- CRITICAL: Fabricated entities, programmes, or phantom record citations

Return ONLY the JSON object, no markdown fences, no preamble."""

print("Sanctions evaluation prompts defined.")

In [ ]:
# ======================================================================
# RUN SANCTIONS-SPECIFIC EVALUATION
# ======================================================================

sanctions_evals = []

print(f"Running sanctions-specific evaluation on {len(ragas_rows)} queries...")
print()

for i, row in enumerate(ragas_rows, 1):
    qid = row['query_id']
    print(f"  [{i}/{len(ragas_rows)}] {qid}: {row['question'][:50]}...", end=' ')
    
    # Format contexts as numbered list
    contexts_text = '\n'.join(
        f"[Record {j+1}]: {ctx}" 
        for j, ctx in enumerate(row['contexts'])
    )
    
    prompt = SANCTIONS_EVAL_TEMPLATE.format(
        question=row['question'],
        contexts=contexts_text,
        answer=row['answer'],
    )
    
    try:
        t0 = time.time()
        response = eval_llm.invoke([
            {"role": "system", "content": SANCTIONS_EVAL_SYSTEM},
            {"role": "user", "content": prompt},
        ])
        latency = time.time() - t0
        
        # Parse JSON response
        raw = response.content.strip()
        # Strip markdown fences if present
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'\s*```$', '', raw)
        
        eval_data = json.loads(raw)
        eval_data['query_id'] = qid
        eval_data['eval_latency_sec'] = round(latency, 1)
        sanctions_evals.append(eval_data)
        
        risk = eval_data.get('risk_level', '?')
        overall = eval_data.get('overall_sanctions_quality', 0)
        print(f"Quality={overall:.2f} Risk={risk} ({latency:.1f}s)")
        
    except json.JSONDecodeError as e:
        print(f"JSON parse error: {e}")
        sanctions_evals.append({
            'query_id': qid,
            'overall_sanctions_quality': None,
            'risk_level': 'EVAL_ERROR',
            'summary': f'JSON parse error: {str(e)[:100]}',
            'raw_response': raw[:500],
        })
    except Exception as e:
        print(f"Error: {e}")
        sanctions_evals.append({
            'query_id': qid,
            'overall_sanctions_quality': None,
            'risk_level': 'EVAL_ERROR',
            'summary': f'Error: {str(e)[:100]}',
        })

print(f"\nSanctions evaluation complete: {len(sanctions_evals)} queries evaluated.")

---
## 9 · Combined Evaluation Summary

In [ ]:
# ======================================================================
# COMBINED SUMMARY — RAGAS + Sanctions-Specific
# ======================================================================

df_sanctions = pd.DataFrame(sanctions_evals)

# Merge RAGAS scores with sanctions scores
if not df_ragas.empty and not df_sanctions.empty:
    df_combined = df_ragas.merge(df_sanctions, on='query_id', how='outer')
elif not df_sanctions.empty:
    df_combined = df_sanctions.copy()
else:
    df_combined = df_ragas.copy()

print("=" * 80)
print("  PHASE 09 — RAGAS + SANCTIONS EVALUATION SUMMARY")
print("=" * 80)
print()

# RAGAS aggregate
print("RAGAS METRICS (aggregate):")
print("-" * 40)
ragas_metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
for m in ragas_metrics:
    if m in df_combined.columns:
        vals = df_combined[m].dropna()
        if len(vals) > 0:
            print(f"  {m:25s}: {vals.mean():.4f}  (min={vals.min():.4f}, max={vals.max():.4f})")
        else:
            print(f"  {m:25s}: N/A")
    else:
        print(f"  {m:25s}: Not computed")

print()

# Sanctions aggregate
print("SANCTIONS-SPECIFIC METRICS (aggregate):")
print("-" * 40)
sanctions_metrics = [
    'entity_attribution_score',
    'programme_grounding_score',
    'classification_justification_score',
    'overall_sanctions_quality',
]
for m in sanctions_metrics:
    if m in df_combined.columns:
        vals = pd.to_numeric(df_combined[m], errors='coerce').dropna()
        if len(vals) > 0:
            print(f"  {m:40s}: {vals.mean():.4f}  (min={vals.min():.4f}, max={vals.max():.4f})")
        else:
            print(f"  {m:40s}: N/A")

print()

# Risk distribution
if 'risk_level' in df_combined.columns:
    print("RISK LEVEL DISTRIBUTION:")
    print("-" * 40)
    risk_counts = df_combined['risk_level'].value_counts()
    for level in ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL', 'EVAL_ERROR']:
        count = risk_counts.get(level, 0)
        bar = '█' * count
        print(f"  {level:12s}: {count:3d}  {bar}")

print()

# Fabrication check
if 'fabrication_detected' in df_combined.columns:
    fab_count = df_combined['fabrication_detected'].sum()
    print(f"FABRICATION DETECTED: {fab_count} / {len(df_combined)} queries")
    if fab_count > 0:
        print("  ⚠ CRITICAL: Review fabricated claims in the detail below.")
        fab_rows = df_combined[df_combined['fabrication_detected'] == True]
        for _, row in fab_rows.iterrows():
            print(f"    Query {row['query_id']}: {row.get('fabricated_claims', [])}")

print(f"\n{'='*80}")

---
## 10 · Per-Query Detail View

In [ ]:
# ======================================================================
# Per-query detail — change VIEW_INDEX to browse
# ======================================================================

VIEW_INDEX = 0  # Change this to view different queries

if not df_combined.empty and VIEW_INDEX < len(df_combined):
    row = df_combined.iloc[VIEW_INDEX]
    print(f"{'='*80}")
    print(f"QUERY: {row.get('query_id', '?')}")
    print(f"{'='*80}")
    print()
    
    print("RAGAS Scores:")
    for m in ragas_metrics:
        val = row.get(m, 'N/A')
        if isinstance(val, float):
            print(f"  {m:25s}: {val:.4f}")
        else:
            print(f"  {m:25s}: {val}")
    
    print()
    print("Sanctions Evaluation:")
    for m in sanctions_metrics:
        val = row.get(m, 'N/A')
        if isinstance(val, float):
            print(f"  {m:40s}: {val:.4f}")
        else:
            print(f"  {m:40s}: {val}")
    
    print(f"\n  Risk Level: {row.get('risk_level', 'N/A')}")
    print(f"  Fabrication: {row.get('fabrication_detected', 'N/A')}")
    print(f"  Summary: {row.get('summary', 'N/A')}")
    
    # Show issues if any
    for issue_col in ['entity_attribution_issues', 'programme_grounding_issues',
                      'classification_justification_issues', 'fabricated_claims']:
        issues = row.get(issue_col, [])
        if issues and isinstance(issues, list) and len(issues) > 0:
            print(f"\n  {issue_col}:")
            for iss in issues:
                print(f"    - {iss}")
    
    print(f"\n{'='*80}")
else:
    print("No results to display.")

---
## 11 · Export Results

Exports:
1. **Updated enriched Excel** — RAGAS + sanctions scores merged back into the original data
2. **Evaluation summary CSV** — per-query scores for easy comparison
3. **Full evaluation JSON** — all details including issue lists

In [ ]:
# ======================================================================
# EXPORT
# ======================================================================

output_dir = Path('../data/ragas_output')
output_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# --- 1. Merge scores back into original DataFrame ---
# Build query-level score lookup
score_lookup = {}
for _, row in df_combined.iterrows():
    qid = row.get('query_id', '')
    score_lookup[qid] = {
        'ragas_faithfulness': row.get('faithfulness', None),
        'ragas_answer_relevancy': row.get('answer_relevancy', None),
        'ragas_context_precision': row.get('context_precision', None),
        'ragas_context_recall': row.get('context_recall', None),
        'sanctions_entity_attribution': row.get('entity_attribution_score', None),
        'sanctions_programme_grounding': row.get('programme_grounding_score', None),
        'sanctions_classification_quality': row.get('classification_justification_score', None),
        'sanctions_overall_quality': row.get('overall_sanctions_quality', None),
        'sanctions_risk_level': row.get('risk_level', None),
        'sanctions_fabrication_detected': row.get('fabrication_detected', None),
        'sanctions_eval_summary': row.get('summary', None),
    }

# Add columns to original dataframe
df_final = df.copy()
new_cols = list(list(score_lookup.values())[0].keys()) if score_lookup else []

for col in new_cols:
    df_final[col] = df_final['query_id'].map(
        {qid: scores.get(col) for qid, scores in score_lookup.items()}
    )

# --- 2. Export enriched Excel ---
excel_path = output_dir / f'ragas_enriched_{timestamp}.xlsx'

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    # Main data sheet
    df_final.to_excel(writer, index=False, sheet_name='rag_ragas_enriched')
    ws = writer.sheets['pooling_rag_ragas']
    ws.freeze_panes = 'A2'
    
    # Per-query RAGAS summary sheet
    summary_cols = ['query_id'] + ragas_metrics + sanctions_metrics + ['risk_level', 'fabrication_detected', 'summary']
    available = [c for c in summary_cols if c in df_combined.columns]
    df_combined[available].to_excel(writer, index=False, sheet_name='ragas_summary')
    ws2 = writer.sheets['ragas_summary']
    ws2.freeze_panes = 'A2'
    for col_idx in range(1, len(available) + 1):
        ws2.column_dimensions[ws2.cell(row=1, column=col_idx).column_letter].width = 22

print(f"Excel exported: {excel_path}")

# --- 3. Export evaluation CSV ---
csv_path = output_dir / f'ragas_scores_{timestamp}.csv'
df_combined.to_csv(csv_path, index=False)
print(f"CSV exported  : {csv_path}")

# --- 4. Export full JSON ---
json_path = output_dir / f'ragas_full_eval_{timestamp}.json'
json_export = {
    'metadata': {
        'eval_model': EVAL_MODEL,
        'timestamp': timestamp,
        'source_file': str(rag_path),
        'n_queries': len(ragas_rows),
    },
    'ragas_aggregate': {k: v for k, v in (ragas_result.items() if ragas_result else {}.items()) if isinstance(v, (int, float))},
    'sanctions_aggregate': {
        m: float(pd.to_numeric(df_combined[m], errors='coerce').mean())
        for m in sanctions_metrics if m in df_combined.columns
    },
    'per_query': sanctions_evals,
}

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(json_export, f, indent=2, ensure_ascii=False, default=str)
print(f"JSON exported : {json_path}")

print(f"\n{'='*60}")
print(f"  ALL EXPORTS COMPLETE")
print(f"{'='*60}")
print(f"  Output dir: {output_dir}/")
print(f"  Files:")
for f in sorted(output_dir.glob('*')):
    print(f"    {f.name}")

---
## 12 · Cleanup — Purge API Key from Memory

Final security step: remove the API key from the environment variable.
Run this when you are done.

In [ ]:
# ======================================================================
# SECURITY CLEANUP — Remove API key from environment
# ======================================================================

if 'ANTHROPIC_API_KEY' in os.environ:
    del os.environ['ANTHROPIC_API_KEY']
    print("API key removed from environment.")
else:
    print("API key was already cleared.")

print("\nSession cleanup complete. You may safely close this notebook.")
print("To re-run evaluations, restart the kernel and enter the key again.")

---
## 13 · Next Steps

1. **Review risk flags** — Any query with `CRITICAL` or `HIGH` risk should be
   manually reviewed by an analyst before relying on the RAG output.

2. **Low faithfulness scores** (< 0.7) indicate the LLM may have hallucinated
   sanctions data — cross-check those triage notes against the source records.

3. **Programme grounding issues** suggest the LLM may have cited sanctions
   programmes not present in the retrieved data — a serious compliance risk.

4. **Compare models** — Run Notebook 08 with a different LLM, then re-run
   this notebook to compare RAGAS scores across models.

5. **Add analyst ground truth** — Have an analyst label each query's expected
   output, then use RAGAS `context_recall` with real ground truth for the
   most rigorous evaluation.

6. **Tune the retrieval** — If `context_precision` is low, the retrieval
   pipeline (Notebooks 01–07) may need retuning to surface better records.